# Long Term Memory: Episodic & Semantic Memory in MongoDB

**Long-term memory** is the persistent store of information that survives beyond a single interaction or task. While short-term memory tracks *what is happening now*, long-term memory captures *what the agent has learned over time* — enabling it to adapt, personalise, and improve across sessions.

The presentation identifies three long-term memory types:

| Type | What it stores | Examples |
|------|---------------|----------|
| **Episodic** | Records of past interactions | Conversations, summarisations |
| **Semantic** | Facts and relationships about the world | Knowledge base, entity memory, persona memory |
| **Procedural** | How to do things | Workflows, toolbox definitions |

This notebook implements **episodic** and **semantic** memory:

1. **Memory ingestion** — the LLM extracts key facts from a conversation and stores them as structured memory documents in MongoDB, encoded with VoyageAI embeddings.
2. **Memory retrieval** — before responding, the agent queries MongoDB with `$vectorSearch` to surface the most relevant past memories and injects them into the context window.
3. **Memory lifecycle** — memories can be updated as facts change and deleted (forgotten) when they become stale.

```
Conversation → LLM extracts facts → embed (VoyageAI) → MongoDB
                                                             ↕
New session → embed query (VoyageAI) → $vectorSearch → inject memories → LLM
```

> **Scenario:** The same travel assistant from notebook 01, but now it remembers user preferences *across* sessions — favourite cities, budget range, amenity requirements — and uses them to give personalised recommendations from the very first message of a new session.

In [ ]:
import { MongoClient } from 'mongodb';
import { ChatAnthropic } from '@langchain/anthropic';
import { tool } from '@langchain/core/tools';
import { z } from 'zod';

// ← Set VOYAGE_API_KEY as a Codespace secret, or paste it here as a fallback
const VOYAGE_API_KEY = process.env.VOYAGE_API_KEY ?? 'pa-...';

// ← Set ANTHROPIC_API_KEY as a Codespace secret, or paste it here as a fallback
const ANTHROPIC_API_KEY = process.env.ANTHROPIC_API_KEY ?? 'sk-ant-...';

const client = new MongoClient(process.env.MONGODB_URI!, { appName: 'devrel-workshop-agentmemory-langchain' });
await client.connect();
const db       = client.db('agent_memory_lab');
const listings = db.collection('listings');
const memories = db.collection('long_term_memories');

console.log('Connected. Listings:', await listings.countDocuments());

## Embedding helper

Memories are encoded as vectors using `voyage-4-large` (for storage) and queried using `voyage-4-lite`. Both models share one embedding space — index with large, query with lite, no re-indexing needed.

In [ ]:
async function embed(texts: string[], model: string, inputType: 'document' | 'query'): Promise<number[][]> {
  const res = await fetch('https://api.voyageai.com/v1/embeddings', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json', 'Authorization': `Bearer ${VOYAGE_API_KEY}` },
    body: JSON.stringify({ input: texts, model, input_type: inputType }),
  });
  if (!res.ok) throw new Error(await res.text());
  const json = await res.json() as { data: { embedding: number[] }[] };
  return json.data.map(d => d.embedding);
}

console.log('Embedding helper ready.');

## Memory document schema

Each memory document captures:
- `userId` — who this memory belongs to
- `type` — `episodic` (from a specific interaction) or `semantic` (a distilled fact)
- `content` — the human-readable memory text
- `embedding` — the VoyageAI vector for similarity search
- `metadata` — tags, source session, timestamps

In [ ]:
interface MemoryDocument {
  userId:    string;
  type:      'episodic' | 'semantic';
  content:   string;
  embedding: number[];
  metadata:  {
    tags:           string[];
    sourceSession?: string;
    createdAt:      Date;
    updatedAt:      Date;
  };
}

console.log('Schema defined.');

## LLM-powered memory extraction

After a conversation ends, the LLM reviews the transcript and extracts distinct, reusable facts. This is the **Memory Ingestion & Curation** phase — transforming passive conversation data into active, retrievable memory stored in MongoDB.

In [ ]:
const llm = new ChatAnthropic({ model: 'claude-haiku-4-5-20251001', apiKey: ANTHROPIC_API_KEY });

async function extractMemories(conversation: string, userId: string, sessionId: string): Promise<void> {
  const prompt = `You are a memory extraction system. Given the conversation below, extract 3-5 distinct, 
reusable facts about the user's travel preferences, requirements, or history. 
Return ONLY a JSON array of strings. Each string is one self-contained fact. No explanation.

Conversation:
${conversation}`;

  const response = await llm.invoke(prompt);
  const raw = (response.content as string).replace(/```json\n?|```/g, '').trim();
  const facts: string[] = JSON.parse(raw);

  const embeddings = await embed(facts, 'voyage-4-large', 'document');

  const docs: MemoryDocument[] = facts.map((fact, i) => ({
    userId,
    type: 'semantic',
    content: fact,
    embedding: embeddings[i],
    metadata: {
      tags: ['travel', 'preference'],
      sourceSession: sessionId,
      createdAt: new Date(),
      updatedAt: new Date(),
    },
  }));

  await memories.insertMany(docs);
  console.log(`Stored ${docs.length} memories for user "${userId}"`);
  facts.forEach((f, i) => console.log(`  [${i + 1}] ${f}`));
}

console.log('Memory extraction function ready.');

## Ingesting memories from a past session

Simulate a conversation that already happened in a previous session. The LLM extracts the key facts and stores them — embedded with VoyageAI — in MongoDB.

In [ ]:
const pastConversation = `
User: Hi, I am looking for a place in Barcelona under $150/night.
Agent: I found a few options. There is a Studio in Barcelona at $98/night and an Apartment at $130/night.
User: I need at least 2 bedrooms — we are two people traveling together.
Agent: Got it. I found a 2BR Apartment in Barcelona at $145/night with WiFi and a Kitchen.
User: Perfect! We also need parking. And we prefer quiet neighbourhoods over city centre.
Agent: I found a 2BR Condo with Parking at $140/night in Eixample.
User: Great, we love Barcelona! We will probably visit again next summer.
`;

await extractMemories(pastConversation, 'user-alice', 'session-barcelona-2024');

## Creating the vector search index on memories

To retrieve memories by semantic similarity we need a `$vectorSearch` index on the `embedding` field. The `userId` and `type` filter fields enable per-user scoping and memory-type filtering at query time.

In [ ]:
const MEMORY_INDEX = 'memory_vector_index';

try {
  await memories.dropSearchIndex(MEMORY_INDEX);
  await new Promise(r => setTimeout(r, 2000));
} catch { /* index did not exist */ }

await memories.createSearchIndex({
  name: MEMORY_INDEX,
  type: 'vectorSearch',
  definition: {
    fields: [
      { type: 'vector', path: 'embedding', numDimensions: 1024, similarity: 'cosine' },
      { type: 'filter', path: 'userId' },
      { type: 'filter', path: 'type' },
    ],
  },
});

console.log('Waiting for memory index to be READY...');
for (let i = 0; i < 30; i++) {
  await new Promise(r => setTimeout(r, 2000));
  const [idx] = await memories.listSearchIndexes(MEMORY_INDEX).toArray();
  console.log(' status:', idx?.status);
  if (idx?.status === 'READY') break;
}

## Retrieving relevant memories with $vectorSearch

When the agent receives a new query, it embeds it with `voyage-4-lite` and uses `$vectorSearch` to find the most relevant memories. The `userId` filter ensures only this user's memories are considered.

In [ ]:
async function retrieveMemories(query: string, userId: string, limit = 3): Promise<MemoryDocument[]> {
  const [qVec] = await embed([query], 'voyage-4-lite', 'query');

  const results = await memories.aggregate([
    {
      $vectorSearch: {
        index:         MEMORY_INDEX,
        path:          'embedding',
        queryVector:   qVec,
        numCandidates: 20,
        limit,
        filter:        { userId },
      },
    },
    {
      $project: {
        content:  1,
        type:     1,
        metadata: 1,
        score:    { $meta: 'vectorSearchScore' },
      },
    },
  ]).toArray();

  return results as unknown as MemoryDocument[];
}

console.log('Memory retrieval function ready.');

## Testing retrieval — a new session begins

Alice returns with a new message. The agent retrieves her stored memories before responding, even though this is a brand-new session with no prior turns.

In [ ]:
const newQuery = 'I would like to find a place to stay in Europe this summer.';

const retrieved = await retrieveMemories(newQuery, 'user-alice');

console.log(`Retrieved ${retrieved.length} memories for Alice:\n`);
retrieved.forEach((m: any, i: number) => {
  console.log(`[${i + 1}] (score: ${m.score?.toFixed(3)}) ${m.content}`);
});

## Memory-augmented agent response

Retrieved memories are injected into the system prompt before the LLM generates a response. This is the **context engineering** loop from the presentation: memory feeds the context window, and the LLM's output can become new memories.

In [ ]:
async function respondWithMemory(userId: string, userMessage: string): Promise<string> {
  const relevantMemories = await retrieveMemories(userMessage, userId);

  const memoryContext = relevantMemories.length > 0
    ? 'What you remember about this user:\n' +
      relevantMemories.map((m: any) => `- ${m.content}`).join('\n')
    : 'No prior memories about this user.';

  const systemPrompt =
    'You are a personalised travel assistant. Use the user memories below to give tailored recommendations. ' +
    'Always acknowledge relevant preferences without asking the user to repeat them.\n\n' +
    memoryContext;

  const response = await llm.invoke([
    { role: 'user', content: `${systemPrompt}\n\nUser message: ${userMessage}` },
  ]);

  return response.content as string;
}

const response = await respondWithMemory('user-alice', newQuery);
console.log('Agent (memory-augmented):\n', response);

## Contrast — the same question without memory

Without memory injection, the agent gives a generic response and cannot personalise at all.

In [ ]:
const genericResponse = await llm.invoke([
  { role: 'user', content: `You are a travel assistant.\n\n${newQuery}` },
]);

console.log('Agent (no memory):\n', genericResponse.content);

## Storing an episodic memory — a completed booking

Episodic memories record specific past events. Here we store a booking confirmation as a factual episode that the agent can recall in future sessions.

In [ ]:
const episodicFact = 'Alice stayed at the Eixample Condo in Barcelona in July 2024 and rated it 5 stars.';
const [episodicVec] = await embed([episodicFact], 'voyage-4-large', 'document');

const episodicDoc: MemoryDocument = {
  userId:    'user-alice',
  type:      'episodic',
  content:   episodicFact,
  embedding: episodicVec,
  metadata:  {
    tags:           ['booking', 'barcelona', 'completed'],
    sourceSession:  'booking-2024-07',
    createdAt:      new Date('2024-07-15'),
    updatedAt:      new Date('2024-07-15'),
  },
};

await memories.insertOne(episodicDoc);
console.log('Episodic memory stored:', episodicFact);

## Updating a memory — preferences evolve

The user's budget has increased. We update the relevant memory document and re-embed the new content with VoyageAI.

In [ ]:
const updatedFact = 'Alice has a travel budget of up to $200 per night.';
const [updatedVec] = await embed([updatedFact], 'voyage-4-large', 'document');

const updateResult = await memories.updateOne(
  { userId: 'user-alice', content: { $regex: 'budget', $options: 'i' } },
  {
    $set: {
      content:              updatedFact,
      embedding:            updatedVec,
      'metadata.updatedAt': new Date(),
    },
  }
);

console.log(`Updated ${updateResult.modifiedCount} memory document(s).`);
console.log('New content:', updatedFact);

## Forgetting — removing stale or invalid memories

A memory that is no longer valid should be explicitly removed so it does not pollute future context. This is the **Forget** step in the memory engineering lifecycle from the presentation.

In [ ]:
const beforeCount = await memories.countDocuments({ userId: 'user-alice' });
console.log('Memories before forget:', beforeCount);

const deleteResult = await memories.deleteMany({
  userId: 'user-alice',
  'metadata.tags': 'completed',
});

const afterCount = await memories.countDocuments({ userId: 'user-alice' });
console.log(`Deleted ${deleteResult.deletedCount} completed-episode memories.`);
console.log('Memories after forget:', afterCount);

## Cross-user isolation — Bob has no memories

A different `userId` returns zero results. The `userId` filter in `$vectorSearch` enforces per-user memory isolation at the database level.

In [ ]:
const bobMemories = await retrieveMemories('I want a place in Europe', 'user-bob');
console.log(`Memories retrieved for user-bob: ${bobMemories.length}`);
console.log('→ Bob starts with a clean slate — no prior preferences loaded into context.');

In [ ]:
await memories.deleteMany({ userId: 'user-alice' });
try { await memories.dropSearchIndex(MEMORY_INDEX); } catch {}
await client.close();
console.log('Done.');